# Filelock Package Examples

This notebook demonstrates 5 basic, versatile, and comprehensive examples of the `filelock` package.

Filelock provides a platform-independent file locking mechanism.


## Example 1: Basic file locking with FileLock

In [ ]:
from filelock import FileLock
import tempfile
import os
import time

# Create a temporary file for locking
temp_dir = tempfile.gettempdir()
lock_file = os.path.join(temp_dir, "my_app.lock")

print(f"Using lock file: {lock_file}")

# Basic file locking
lock = FileLock(lock_file)

print("Attempting to acquire lock...")
with lock:
    print("✓ Lock acquired successfully!")
    print(f"Lock file exists: {os.path.exists(lock_file)}")
    
    # Do some work while holding the lock
    time.sleep(0.1)
    print("Work completed while holding lock")

print(f"Lock released. Lock file exists: {os.path.exists(lock_file)}")

# Test lock properties
print(f"\nLock file path: {lock.lock_file}")
print(f"Lock is locked: {lock.is_locked}")

# Verify filelock functionality
print(f"FileLock class available: {FileLock.__name__}")
print(f"Lock object type: {type(lock).__name__}")

## Example 2: Timeout handling and non-blocking locks

In [ ]:
from filelock import FileLock, Timeout
import tempfile
import os
import threading
import time

# Create lock file
lock_file = os.path.join(tempfile.gettempdir(), "timeout_test.lock")
lock = FileLock(lock_file)

# Test timeout behavior
print("Testing timeout behavior:")

# Events, not sleeps: a fixed sleep races on a loaded host, and the timeout below must be deterministic
lock_held = threading.Event()
attempt_done = threading.Event()

def hold_lock_briefly():
    """Function to hold lock in background thread until the main thread's timed attempt is over"""
    with FileLock(lock_file):
        print("Background thread acquired lock")
        lock_held.set()
        attempt_done.wait(timeout=10)
        print("Background thread releasing lock")

# Start background thread to hold lock
thread = threading.Thread(target=hold_lock_briefly)
thread.start()
assert lock_held.wait(timeout=10), "background thread never acquired the lock"

# Try to acquire lock with timeout: the lock is held, so this MUST time out
print("Main thread attempting to acquire lock with 0.2s timeout...")
try:
    lock.acquire(timeout=0.2)
except Timeout:
    print("✓ Expected: Timeout occurred - lock was held by another thread")
else:
    lock.release()
    raise AssertionError("main thread acquired a lock the background thread was holding")
finally:
    attempt_done.set()

# Wait for background thread and try again
thread.join()
print("Background thread finished")

# Now acquire lock successfully (a Timeout here propagates and fails the cell)
with lock.acquire(timeout=1.0):
    print("✓ Main thread successfully acquired lock after background thread finished")

# Test non-blocking acquire
print("\nTesting non-blocking acquire:")
try:
    acquired = lock.acquire(timeout=0)  # Non-blocking
    if acquired:
        print("✓ Non-blocking acquire succeeded")
        lock.release()
    else:
        print("Non-blocking acquire failed (lock was busy)")
except Timeout:
    print("Non-blocking acquire timed out immediately (expected if lock was busy)")

## Example 3: Different lock types and platforms

In [ ]:
from filelock import FileLock, SoftFileLock, UnixFileLock
import tempfile
import os
import platform

print(f"Platform: {platform.system()}")
temp_dir = tempfile.gettempdir()

# Test different lock types
lock_types = [
    ("FileLock", FileLock),
    ("SoftFileLock", SoftFileLock),
]

# Add UnixFileLock on Unix-like systems
if platform.system() != "Windows":
    try:
        lock_types.append(("UnixFileLock", UnixFileLock))
    except ImportError:
        print("UnixFileLock not available")

for lock_name, lock_class in lock_types:
    print(f"\nTesting {lock_name}:")
    lock_file = os.path.join(temp_dir, f"{lock_name.lower()}_test.lock")
    
    try:
        lock = lock_class(lock_file)
        
        with lock:
            print(f"✓ {lock_name} acquired and released successfully")
            print(f"  Lock file: {lock.lock_file}")
            print(f"  Is locked: {lock.is_locked}")
        
        print(f"  Final state - Is locked: {lock.is_locked}")
        
    except Exception as e:
        print(f"✗ Error with {lock_name}: {e}")
        raise

# Test lock file cleanup
print("\nTesting lock file cleanup:")
cleanup_lock_file = os.path.join(temp_dir, "cleanup_test.lock")
cleanup_lock = FileLock(cleanup_lock_file)

with cleanup_lock:
    lock_exists_during = os.path.exists(cleanup_lock_file)
    print(f"Lock file exists during lock: {lock_exists_during}")

lock_exists_after = os.path.exists(cleanup_lock_file)
print(f"Lock file exists after release: {lock_exists_after}")

# Verify all filelock classes are importable
available_classes = [cls.__name__ for name, cls in lock_types]
print(f"\nAvailable lock classes: {', '.join(available_classes)}")

## Example 4: Error handling and edge cases

In [ ]:
from filelock import FileLock, Timeout
import tempfile
import os
import threading
import time

print("Testing error handling and edge cases:")

# Test lock file paths. A MISSING parent directory is not an invalid path: filelock creates it
# (ensure_directory_exists -> mkdir(parents=True); measured, filelock 3.32.6). A path whose parent is a
# regular FILE can never be created, by any user, so that one must fail.
with tempfile.TemporaryDirectory() as scratch_dir:
    missing_parent_path = os.path.join(scratch_dir, "nonexistent", "directory", "test.lock")
    with FileLock(missing_parent_path).acquire(timeout=0.1):
        assert os.path.isdir(os.path.dirname(missing_parent_path)), "filelock did not create the missing parent directory"
        print("✓ Missing parent directory created and lock acquired")

    regular_file = os.path.join(scratch_dir, "regular_file")
    open(regular_file, "w").close()
    invalid_path = os.path.join(regular_file, "test.lock")
    try:
        with FileLock(invalid_path).acquire(timeout=0.1):
            pass
    except (FileExistsError, NotADirectoryError) as e:
        print(f"✓ Correctly rejected a lock path under a regular file: {type(e).__name__}")
    else:
        raise AssertionError(f"FileLock acquired a lock at an impossible path: {invalid_path}")

# Test lock acquisition patterns
temp_dir = tempfile.gettempdir()
test_lock_file = os.path.join(temp_dir, "edge_case_test.lock")
lock = FileLock(test_lock_file)

# Test multiple acquire/release cycles
print("\nTesting multiple acquire/release cycles:")
for i in range(3):
    try:
        with lock:
            print(f"✓ Cycle {i+1}: Lock acquired successfully")
            time.sleep(0.01)
    except Exception as e:
        print(f"✗ Cycle {i+1}: Error - {e}")
        raise

# Test concurrent access simulation
print("\nTesting concurrent access simulation:")
results = []

def try_acquire_lock(thread_id, results_list):
    """Try to acquire lock from multiple threads"""
    try:
        thread_lock = FileLock(test_lock_file)
        with thread_lock.acquire(timeout=0.1):
            results_list.append(f"Thread {thread_id}: SUCCESS")
            time.sleep(0.05)
    except Timeout:
        results_list.append(f"Thread {thread_id}: TIMEOUT")
    except Exception as e:
        results_list.append(f"Thread {thread_id}: ERROR - {e}")

# Start multiple threads trying to acquire the same lock
threads = []
for i in range(3):
    thread = threading.Thread(target=try_acquire_lock, args=(i, results))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

# Check results
successes = sum(1 for r in results if "SUCCESS" in r)
timeouts = sum(1 for r in results if "TIMEOUT" in r)
print(f"Results: {successes} successes, {timeouts} timeouts")
for result in results:
    print(f"  {result}")

# Verify filelock error handling works
print(f"\nFilelock module handles errors properly: {Timeout.__name__ == 'Timeout'}")

## Example 5: Advanced usage patterns and performance

In [ ]:
from filelock import FileLock
import tempfile
import os
import time
import threading
from contextlib import contextmanager

# Custom lock manager context
@contextmanager
def timed_lock(lock_file, timeout=5.0):
    """Context manager that times lock acquisition"""
    lock = FileLock(lock_file)
    start_time = time.time()
    try:
        with lock.acquire(timeout=timeout):
            acquire_time = time.time() - start_time
            print(f"Lock acquired in {acquire_time:.4f} seconds")
            yield lock
    finally:
        total_time = time.time() - start_time
        print(f"Total lock duration: {total_time:.4f} seconds")

# Test custom lock manager
temp_dir = tempfile.gettempdir()
performance_lock = os.path.join(temp_dir, "performance_test.lock")

print("Testing custom lock manager:")
with timed_lock(performance_lock, timeout=2.0) as lock:
    print(f"Working with lock: {lock.is_locked}")
    time.sleep(0.1)

# Performance test - rapid lock/unlock cycles
print("\nPerformance test - rapid lock cycles:")
rapid_lock_file = os.path.join(temp_dir, "rapid_test.lock")
rapid_lock = FileLock(rapid_lock_file)

num_cycles = 50
start_time = time.time()

for i in range(num_cycles):
    with rapid_lock:
        pass  # Just acquire and release

total_time = time.time() - start_time
avg_time = total_time / num_cycles * 1000  # Convert to milliseconds

print(f"Completed {num_cycles} lock cycles in {total_time:.4f} seconds")
print(f"Average time per cycle: {avg_time:.2f} ms")

# Test lock with different file operations
print("\nTesting lock with file operations:")
data_file = os.path.join(temp_dir, "shared_data.txt")
data_lock_file = os.path.join(temp_dir, "data_file.lock")
data_lock = FileLock(data_lock_file)

# Write data safely
with data_lock:
    with open(data_file, 'w') as f:
        f.write("Safely written data\n")
        f.write(f"Timestamp: {time.time()}\n")
    print("✓ Data written while holding lock")

# Read data safely
with data_lock:
    if os.path.exists(data_file):
        with open(data_file, 'r') as f:
            content = f.read().strip()
        print(f"✓ Data read while holding lock: {len(content)} characters")

# Cleanup test files
try:
    os.unlink(data_file)
    print("✓ Test files cleaned up")
except OSError:
    pass  # File might not exist

# Final verification
print(f"\nFilelock module fully functional: {hasattr(FileLock, 'acquire') and hasattr(FileLock, 'release')}")
print("\nFilelock testing completed successfully!")